In [3]:
import pandas as pd
import numpy as np
import requests
import json
import os
import re
import joblib

from jsonschema import validate, ValidationError

In [4]:
df = pd.read_csv("cleaned_data.csv")

print("Dataset Shape:", df.shape)
df.head()


Dataset Shape: (20010, 23)


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,signup_date,churn,internal_flag
0,17267,female,0.0,No,Yes,41.0,Yes,Yes,Fiber optic,Yes,...,No,No,One year,Yes,Mailed check,53.77,848.51,2024-08-02,No,A
1,9242,Female,0.0,Yes,No,27.0,Yes,Yes,Fiber optic,Yes,...,No,No,Month-to-month,Yes,Credit card,84.76,6916.82,2022-07-15,No,A
2,19107,Male,1.0,Yes,Yes,36.0,Yes,Yes,DSL,No,...,No,Yes,Month-to-month,Y,Mailed check,28.61,1099.85,2024-07-17,No,A
3,8986,Female,0.0,No,No,1.0,N,N,No,No,...,Yes,Yes,Month-to-month,No,Electronic check,119.07,6943.98,2024-06-04,No,A
4,2867,Male,0.0,No,Yes,20.0,Yes,Yes,fiber,No,...,N,Yes,Month-to-month,No,Electronic check,108.74,5279.77,2023-04-30,No,A


In [5]:
model = joblib.load("best_model.pkl")

print("Best model loaded successfully!")
print(model)

Best model loaded successfully!
Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(min_samples_leaf=5, random_state=42))])


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20010 entries, 0 to 20009
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        20010 non-null  int64  
 1   gender             20010 non-null  object 
 2   senior_citizen     20010 non-null  float64
 3   partner            20010 non-null  object 
 4   dependents         20010 non-null  object 
 5   tenure             20010 non-null  float64
 6   phone_service      20010 non-null  object 
 7   multiple_lines     20010 non-null  object 
 8   internet_service   20010 non-null  object 
 9   online_security    20010 non-null  object 
 10  online_backup      20010 non-null  object 
 11  device_protection  20010 non-null  object 
 12  tech_support       20010 non-null  object 
 13  streaming_tv       20010 non-null  object 
 14  streaming_movies   20010 non-null  object 
 15  contract           20010 non-null  object 
 16  paperless_billing  200

In [7]:
# Prepare input data like Part 2

X = df.drop(["monthly_charges", "customer_id"], axis=1)

# Convert signup_date into year and month
X["signup_date"] = pd.to_datetime(X["signup_date"])
X["signup_year"] = X["signup_date"].dt.year
X["signup_month"] = X["signup_date"].dt.month
X = X.drop("signup_date", axis=1)

# One-hot encoding
categorical_cols = X.select_dtypes(include="object").columns

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

print(X.shape)
X.head()

(20010, 66)


,senior_citizen,tenure,total_charges,signup_year,signup_month,gender_Female,gender_M,gender_Male,gender_female,gender_male,...,paperless_billing_No,paperless_billing_Y,paperless_billing_Yes,payment_method_Credit card,payment_method_E-check,payment_method_Electronic check,payment_method_Mailed check,payment_method_e-check,churn_Yes,internal_flag_B
0,0.0,41.0,848.51,2024,8,False,False,False,True,False,...,False,False,True,False,False,False,True,False,False,False
1,0.0,27.0,6916.82,2022,7,True,False,False,False,False,...,False,False,True,True,False,False,False,False,False,False
2,1.0,36.0,1099.85,2024,7,False,False,True,False,False,...,False,True,False,False,False,False,True,False,False,False
3,0.0,1.0,6943.98,2024,6,True,False,False,False,False,...,True,False,False,False,False,True,False,False,False,False
4,0.0,20.0,5279.77,2023,4,False,False,True,False,False,...,True,False,False,False,False,True,False,False,False,False


In [8]:
# Match model feature order
X = X.reindex(columns=model.feature_names_in_, fill_value=0)

print(X.shape)


(20010, 66)


In [9]:
sample = X.iloc[[0]]

prediction = model.predict(sample)
probability = model.predict_proba(sample)

print("Prediction:", prediction[0])
print("Probability:", probability[0])

Prediction: 0
Probability: [0.6670759 0.3329241]


In [4]:
!pip install python-dotenv

In [6]:
from dotenv import load_dotenv
import os
import requests

# Load variables from .env
load_dotenv()

# Read API key
api_key = os.getenv("LLM_API_KEY")

if api_key is None:
    raise ValueError("LLM_API_KEY not found in .env file")

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

In [11]:
system_prompt = "You are a helpful assistant."

user_prompt = "Reply with only the word: hello"

result = call_llm(system_prompt, user_prompt)

print(result)

hello


In [12]:
schema = {
    "type": "object",
    "properties": {
        "prediction": {"type": "string"},
        "probability": {"type": "number"},
        "reason": {"type": "string"},
        "risk_level": {"type": "string"},
        "recommendation": {"type": "string"}
    },
    "required": [
        "prediction",
        "probability",
        "reason",
        "risk_level",
        "recommendation"
    ]
}

print("Schema created successfully!")

Schema created successfully!


In [13]:
fallback = {
    "prediction": None,
    "probability": None,
    "reason": None,
    "risk_level": None,
    "recommendation": None
}




In [14]:
def validate_response(response_text):

    try:
        data = json.loads(response_text)
    except json.JSONDecodeError as e:
        print("JSON Decode Error:", e)
        return fallback

    try:
        validate(instance=data, schema=schema)
        print(" JSON Validation Passed")
        return data

    except ValidationError as e:
        print("Validation Error:", e.message)
        return fallback

In [15]:
test_json = """
{
    "prediction": "No Churn",
    "probability": 0.61,
    "reason": "Customer has a stable contract and long tenure.",
    "risk_level": "Low",
    "recommendation": "Continue current service."
}
"""

result = validate_response(test_json)

print(result)

 JSON Validation Passed
{'prediction': 'No Churn', 'probability': 0.61, 'reason': 'Customer has a stable contract and long tenure.', 'risk_level': 'Low', 'recommendation': 'Continue current service.'}


In [16]:
import requests

def call_llm(system_prompt, user_prompt, temperature=0, max_tokens=512):

    response = requests.post(
        url,
        headers=headers,
        json={
            "model": "openai/gpt-4o-mini",
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
    )

    if response.status_code != 200:
        print("API Error:")
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [17]:
import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

In [18]:
text1 = "My email is safina@example.com"
text2 = "Customer has a one year contract."

print("Text 1:", has_pii(text1))
print("Text 2:", has_pii(text2))

Text 1: True
Text 2: False


In [19]:
def safe_call_llm(system_prompt, user_input, temperature=0):

    if has_pii(user_input):
        print("Input blocked: PII detected.")
        return None

    return call_llm(system_prompt, user_input, temperature)

In [20]:
system_prompt = "You are a helpful assistant."

user_input = "My email is safina@example.com"

result = safe_call_llm(system_prompt, user_input)

print(result)


Input blocked: PII detected.
None


In [21]:
system_prompt = "You are a helpful assistant."

user_input = "Customer has a one year contract."

result = safe_call_llm(system_prompt, user_input)

print(result)

It seems like you might be looking for information or assistance related to a one-year contract for a customer. Could you please provide more details about what you need help with? For example, are you looking for advice on contract terms, renewal processes, customer service, or something else?


In [22]:
sample1 = X.iloc[[0]]
sample2 = X.iloc[[1]]
sample3 = X.iloc[[2]]

samples = [sample1, sample2, sample3]

In [23]:
for i, sample in enumerate(samples, start=1):

    prediction = model.predict(sample)[0]
    probability = model.predict_proba(sample)[0][1]

    print(f"\nCustomer {i}")
    print("Prediction:", prediction)
    print("Churn Probability:", round(probability, 4))


Customer 1
Prediction: 0
Churn Probability: 0.3329

Customer 2
Prediction: 1
Churn Probability: 0.5189

Customer 3
Prediction: 0
Churn Probability: 0.4323


In [24]:
system_prompt = """
You are a telecom churn analyst.

Return ONLY valid JSON.

{
    "prediction": "",
    "probability": 0,
    "reason": "",
    "risk_level": "",
    "recommendation": ""
}
"""

In [25]:
#sample 1
prediction = model.predict(sample1)[0]
probability = float(model.predict_proba(sample1)[0][1])

user_prompt = f"""
Prediction: {prediction}
Churn Probability: {probability:.4f}

Explain the prediction.

Return ONLY JSON.
"""

response = safe_call_llm(system_prompt, user_prompt)

validated_result = validate_response(response)

print("Prediction:", validated_result["prediction"])
print("Probability:", validated_result["probability"])
print("Reason:", validated_result["reason"])
print("Risk Level:", validated_result["risk_level"])
print("Recommendation:", validated_result["recommendation"])

 JSON Validation Passed
Prediction: 0
Probability: 0.3329
Reason: The prediction of 0 indicates that the customer is not expected to churn, despite a moderate probability of 33.29%. This suggests that while there is some risk of churn, other factors may indicate loyalty or satisfaction with the service.
Risk Level: medium
Recommendation: Monitor customer engagement and satisfaction levels closely, and consider implementing retention strategies to further reduce churn risk.


In [26]:
#sample 2
prediction = model.predict(sample2)[0]
probability = float(model.predict_proba(sample2)[0][1])

user_prompt = f"""
Prediction: {prediction}
Churn Probability: {probability:.4f}

Explain the prediction.

Return ONLY JSON.
"""

response = safe_call_llm(system_prompt, user_prompt)

validated_result = validate_response(response)

print("Prediction:", validated_result["prediction"])
print("Probability:", validated_result["probability"])
print("Reason:", validated_result["reason"])
print("Risk Level:", validated_result["risk_level"])
print("Recommendation:", validated_result["recommendation"])

 JSON Validation Passed
Prediction: churn
Probability: 0.5189
Reason: The probability of 51.89% indicates that there is a significant chance that the customer will leave the service.
Risk Level: high
Recommendation: Implement retention strategies such as personalized offers or improved customer service to reduce churn risk.


In [27]:
#sample 3
prediction = model.predict(sample3)[0]
probability = float(model.predict_proba(sample3)[0][1])

user_prompt = f"""
Prediction: {prediction}
Churn Probability: {probability:.4f}

Explain the prediction.

Return ONLY JSON.
"""

response = safe_call_llm(system_prompt, user_prompt)

if response is not None:
    validated_result = validate_response(response)
    print(validated_result)

 JSON Validation Passed
{'prediction': '0', 'probability': 0.4323, 'reason': 'The prediction of 0 indicates that the model does not expect the customer to churn, despite a moderate probability of 43.23%. This suggests that while there is a significant chance of churn, other factors may indicate loyalty or satisfaction.', 'risk_level': 'medium', 'recommendation': 'Monitor customer engagement and satisfaction closely, and consider implementing retention strategies to further reduce churn risk.'}


In [28]:
# PII Guardrail Test
user_input = "Customer email: abc@gmail.com"

if has_pii(user_input):
    print("Input blocked: PII detected.")
else:
    response = safe_call_llm(system_prompt, user_input)
    print(response)
    

Input blocked: PII detected.


In [29]:
# Test 2 - Clean Input (Should pass)

user_input = "Prediction: No churn"

if has_pii(user_input):
    print("Input blocked: PII detected.")
else:
    response = safe_call_llm(system_prompt, user_input)
    print(response)

{
    "prediction": "No churn",
    "probability": 0.85,
    "reason": "Customer has a long tenure and consistently pays bills on time.",
    "risk_level": "Low",
    "recommendation": "Continue to engage with the customer through loyalty programs and personalized offers."
}


In [30]:
# Temperature Comparison 0
response_temp0 = safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(response_temp0)

{
    "prediction": "0",
    "probability": 0.4323,
    "reason": "The prediction of 0 indicates that the model does not expect the customer to churn, despite a moderate probability of 43.23%. This suggests that while there is a significant chance of churn, other factors may indicate loyalty or satisfaction.",
    "risk_level": "medium",
    "recommendation": "Monitor customer engagement and satisfaction closely, and consider implementing retention strategies to further reduce churn risk."
}


In [31]:
# Temperature Comparison 0.7
response_temp07 = safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0.7
)

print(response_temp07)

{
    "prediction": "0",
    "probability": 0.4323,
    "reason": "The prediction of '0' indicates that the customer is not expected to churn despite a moderate probability of 43.23%. This suggests that while there is a significant chance of churn, other factors may indicate retention.",
    "risk_level": "Medium",
    "recommendation": "Implement customer engagement strategies to further reduce churn risk and enhance customer satisfaction."
}
